# 🇵🇭 Sugidanon — Text → Hiligaynon Translator

Team Hague · June 25–26, 2026

Type English/Tagalog → get Hiligaynon. Two modes:
- **Neural** (fluent): a Hiligaynon LLM. Needs a GPU runtime + model download.
- **Dictionary** (instant fallback): offline word-by-word, no GPU.

**For GPU:** Runtime → Change runtime type → Hardware accelerator → **GPU**, then Run all.

## 1. Clone repo

In [ ]:
import os
if not os.path.isdir('tinig-sa-liwanag'):
    !git clone https://github.com/Jazztinn/tinig-sa-liwanag.git
%cd tinig-sa-liwanag

## 2. Dictionary mode (works immediately, no install)

In [ ]:
!python scripts/translate_hil.py "Good morning, I went to the market yesterday"

## 3. Neural mode — install + load model once
First run downloads the model (a few minutes). GPU strongly recommended.
Swap `MODEL` for any Hiligaynon LLM from `RESOURCES.md`.

In [ ]:
!pip -q install transformers accelerate torch sentencepiece

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "welyjesch/lfm25-sft-hiligaynon"  # or PLTAT/hiligaynon_llama_3.1_finetuned_lora

print(f"Loading {MODEL} ... (first time downloads weights)")
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
print("Loaded. Device:", model.device)

In [ ]:
def translate(text, max_new_tokens=128):
    prompt = (
        "Translate the following text into Hiligaynon (Ilonggo). "
        "Reply with only the Hiligaynon translation.\n\n"
        f"Text: {text}\nHiligaynon:"
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tok.decode(out[0], skip_special_tokens=True)
    return decoded.split("Hiligaynon:")[-1].strip()

# quick test
for s in ["Good morning, how are you?", "I went to the market yesterday",
          "Thank you very much for your help"]:
    print(f"{s}\n  -> {translate(s)}\n")

## 4. Interactive — type your own

In [ ]:
while True:
    text = input("EN/TL (blank to stop): ").strip()
    if not text:
        break
    print("  HIL:", translate(text), "\n")

## 5. (Optional) Interactive UI with a text box

In [ ]:
import ipywidgets as widgets
from IPython.display import display

box = widgets.Text(description="EN/TL:", layout=widgets.Layout(width="70%"))
out = widgets.Output()

def on_submit(change):
    with out:
        out.clear_output()
        print("HIL:", translate(box.value))

box.on_submit(on_submit)
display(box, out)